# M9.5 · Gürültü modelleri ve restorasyon

Bozulma modeli $g=f+\eta$ ve gürültüyü temizleyen filtreler. Median, merkez ağırlıklı median, uyarlamalı ortalama ve uyarlamalı median kararlarını elle veriyor; saf Python'da median ve uyarlamalı median'ı yazıp OpenCV'nin `medianBlur`'u ile birebir eşleştiriyor, NumPy'da kayan pencerelerle tüm sıra istatistiği filtrelerini ve uyarlamalı ortalama filtreyi döngüsüz kuruyor, PSNR'ı OpenCV ile aynı hesaplıyor, PyTorch'ta küçük bir artık (residual) CNN gürültü gidericiyi eğitip klasik filtreleri PSNR'da geçiyor ve en sonda uyarlamalı median'ı üç pencere boyutunu birden hesaplayıp maskeyle seçerek vektörleştiriyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m9-5-restoration.html](https://m-gonul.github.io/pytorch-lab/m9-5-restoration.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m9_5_restoration.py
CHECKS = {
 "b": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef sp(img, p, seed):\n    rng = np.random.default_rng(seed)\n    out = img.copy()\n    u = rng.random(img.shape)\n    out[u < p / 2] = 0\n    out[(u >= p / 2) & (u < p)] = 255\n    return out\n\n\ndef ref_adaptive(img, smax=7):\n    H, W = img.shape\n    P = np.pad(img, smax // 2, mode=\"edge\").astype(int)\n    out = np.zeros_like(img)\n    for y in range(H):\n        for x in range(W):\n            k = 3\n            while True:\n                r = k // 2\n                win = P[y + smax // 2 - r:y + smax // 2 + r + 1, x + smax // 2 - r:x + smax // 2 + r + 1]\n                zmin, zmax, zmed = win.min(), win.max(), int(np.median(win))\n                if zmin < zmed < zmax:\n                    z = int(img[y, x])\n                    out[y, x] = z if zmin < z < zmax else zmed\n                    break\n                k += 2\n                if k > smax:\n                    out[y, x] = zmed\n                    break\n    return out\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"cv2/numpy kullanılmıyor\", lambda: no_imports(src, {\"cv2\", \"numpy\", \"skimage\", \"scipy\"}))\n    crop = sp(data.camera()[200:248, 220:268], 0.2, 0)\n\n    def med():\n        for k in (3, 5):\n            got = np.array(need(ns, \"median_filter\")(crop.tolist(), k), np.uint8)\n            ensure(np.array_equal(got, cv2.medianBlur(crop, k)), f\"{k}×{k}: cv2.medianBlur ile farklı\")\n    R.test(\"median_filter = cv2.medianBlur (birebir)\", med)\n\n    def cw():\n        got = np.array(need(ns, \"cw_median\")(crop.tolist(), 3, 3), np.uint8)\n        P = np.pad(crop, 1, mode=\"edge\")\n        win = np.lib.stride_tricks.sliding_window_view(P, (3, 3)).reshape(48, 48, 9)\n        want = np.median(np.concatenate([win, np.repeat(crop[..., None], 2, -1)], -1), -1).astype(np.uint8)\n        ensure(np.array_equal(got, want), \"merkez ağırlıklı median\")\n        ensure(np.array_equal(np.array(ns[\"cw_median\"](crop.tolist(), 5, 1), np.uint8), cv2.medianBlur(crop, 5)), \"w = 1 → normal median\")\n    R.test(\"cw_median\", cw)\n\n    def am():\n        dense = sp(data.camera()[260:300, 100:140], 0.5, 1)\n        got = np.array(need(ns, \"adaptive_median\")(dense.tolist()), np.uint8)\n        ensure(np.array_equal(got, ref_adaptive(dense)), \"başvuru ile farklı\")\n        clean = data.camera()[260:300, 100:140]\n        e_am = np.abs(got.astype(int) - clean).mean(); e_m = np.abs(cv2.medianBlur(dense, 3).astype(int) - clean).mean()\n        return f\"%50 tuz-biber: ortalama mutlak hata uyarlamalı {e_am:.1f}, median 3×3 {e_m:.1f}\"\n    R.test(\"adaptive_median (yoğun gürültü)\", am)\n",
 "c": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, no_loops, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    clean = data.camera()\n    rng = np.random.default_rng(0)\n    noisy_sp = clean.copy(); u = rng.random(clean.shape); noisy_sp[u < 0.05] = 0; noisy_sp[(u >= 0.05) & (u < 0.1)] = 255\n\n    def med():\n        for k in (3, 5, 7):\n            ensure(np.array_equal(need(ns, \"median_np\")(noisy_sp, k), cv2.medianBlur(noisy_sp, k)), f\"{k}×{k}\")\n    R.test(\"median_np = cv2.medianBlur (3, 5, 7)\", med)\n\n    def ps():\n        out = cv2.GaussianBlur(noisy_sp, (5, 5), 0)\n        close(need(ns, \"psnr\")(clean, out), cv2.PSNR(clean, out), atol=1e-6)\n        close(ns[\"psnr\"](clean, clean), cv2.PSNR(clean, clean), what=\"özdeş görüntüler\")\n    R.test(\"psnr = cv2.PSNR\", ps)\n\n    def am():\n        sig = np.sqrt(0.001225)\n        noisy = np.clip(np.rint((clean / 255 + rng.normal(0, sig, clean.shape)) * 255), 0, 255).astype(np.uint8)\n        got = need(ns, \"adaptive_mean\")(noisy, 0.001225, 5)\n        g = noisy.astype(np.float64) / 255\n        P = np.pad(g, 2, mode=\"edge\")\n        from scipy import ndimage\n        m = ndimage.uniform_filter(g, 5, mode=\"nearest\"); v = ndimage.uniform_filter(g * g, 5, mode=\"nearest\") - m * m\n        want = np.clip(np.rint((g - np.minimum(0.001225 / np.maximum(v, 1e-12), 1) * (g - m)) * 255), 0, 255).astype(np.uint8)\n        ensure(np.abs(got.astype(int) - want.astype(int)).max() <= 1, \"formül\")\n        rows = [[\"gürültülü\", f\"{cv2.PSNR(clean, noisy):.2f}\"], [\"uyarlamalı ortalama\", f\"{cv2.PSNR(clean, got):.2f}\"],\n                [\"kutu 5×5\", f\"{cv2.PSNR(clean, cv2.blur(noisy, (5, 5))):.2f}\"], [\"Gauss 5×5\", f\"{cv2.PSNR(clean, cv2.GaussianBlur(noisy, (5, 5), 0)):.2f}\"]]\n        R.table(\"Gauss gürültüsü σ² = 0.001225 (alıştırma 4)\", [\"çıkış\", \"PSNR (dB)\"], rows,\n                note=\"σ_η²'yi yanlış (ör. 0.009) verirsen oran çoğu yerde 1'e kırpılır ve filtre kutu filtresine dönüşür: dene.\")\n    R.test(\"adaptive_mean (alıştırma 4)\", am)\n",
 "d": "import numpy as np\nimport cv2\nimport torch\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    def shape():\n        m = need(ns, \"Denoiser\", \"sınıf\")()\n        x = torch.rand(2, 1, 20, 24)\n        ensure(tuple(m(x).shape) == (2, 1, 20, 24), \"aynı boyutta çıkış\")\n        convs = [l for l in m.modules() if isinstance(l, torch.nn.Conv2d)]\n        ensure(len(convs) == 4 and convs[-1].out_channels == 1, \"4 konvolüsyon, son katman 1 kanal\")\n    R.test(\"Denoiser yapısı\", shape)\n\n    def train():\n        tr = torch.tensor(data.camera(), dtype=torch.float32) / 255\n        te = data.coins()[:256, :256].astype(np.float32) / 255\n        sig = 25 / 255\n        m = need(ns, \"train_denoiser\")(tr, sig)\n        noisy = te + sig * np.random.default_rng(5).normal(size=te.shape).astype(np.float32)\n        with torch.no_grad():\n            out = m(torch.tensor(noisy)[None, None])[0, 0].clamp(0, 1).numpy()\n        P = lambda a: 10 * np.log10(1 / np.mean((a - te) ** 2))\n        p_n, p_g, p_m, p_c = P(noisy), P(cv2.GaussianBlur(noisy, (5, 5), 0)), P(cv2.medianBlur((np.clip(noisy, 0, 1) * 255).astype(np.uint8), 5).astype(np.float32) / 255), P(out)\n        ensure(p_c > p_g + 1.0, f\"CNN {p_c:.2f} dB, Gauss {p_g:.2f} dB: en az 1 dB fark bekleniyordu\")\n        R.table(\"coins, σ = 25 (görülmemiş görüntü)\", [\"yöntem\", \"PSNR (dB)\"],\n                [[\"gürültülü\", f\"{p_n:.2f}\"], [\"Gauss 5×5\", f\"{p_g:.2f}\"], [\"median 5×5\", f\"{p_m:.2f}\"], [\"senin CNN (500 adım)\", f\"{p_c:.2f}\"]])\n    R.test(\"öğrenilen gürültü giderici klasik filtreleri geçiyor\", train)\n",
 "e": "import numpy as np\nimport torch\nfrom skimage import data\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef ref_adaptive(img, smax=7):\n    H, W = img.shape\n    P = np.pad(img, smax // 2, mode=\"edge\").astype(int)\n    out = np.zeros_like(img)\n    for y in range(H):\n        for x in range(W):\n            k = 3\n            while True:\n                r = k // 2\n                win = P[y + smax // 2 - r:y + smax // 2 + r + 1, x + smax // 2 - r:x + smax // 2 + r + 1]\n                zmin, zmax, zmed = win.min(), win.max(), int(np.median(win))\n                if zmin < zmed < zmax:\n                    z = int(img[y, x])\n                    out[y, x] = z if zmin < z < zmax else zmed\n                    break\n                k += 2\n                if k > smax:\n                    out[y, x] = zmed\n                    break\n    return out\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"yalnız pencere boyutu döngüsü\", lambda: max_loops(src, 1, func=\"adaptive_median_vec\"))\n    rng = np.random.default_rng(0)\n    for p in (0.1, 0.4, 0.7):\n        def t(p=p):\n            img = data.camera()[100:180, 200:290].copy()\n            u = rng.random(img.shape); img[u < p / 2] = 0; img[(u >= p / 2) & (u < p)] = 255\n            got = need(ns, \"adaptive_median_vec\")(torch.tensor(img)).numpy()\n            ensure(np.array_equal(got, ref_adaptive(img)), f\"gürültü %{int(p * 100)}: başvuru ile farklı\")\n        R.test(f\"%{int(p * 100)} tuz-biber = döngülü başvuru\", t)\n    if \"adaptive_median_vec\" not in ns:\n        return\n    img = data.camera()[:128, :128].copy(); u = rng.random(img.shape); img[u < 0.2] = 0; img[(u >= 0.2) & (u < 0.4)] = 255\n    t_ref = bench(lambda: ref_adaptive(img), repeat=1, number=1)\n    t_vec = bench(lambda: ns[\"adaptive_median_vec\"](torch.tensor(img)), repeat=3)\n    R.table(\"128 × 128, %40 gürültü\", [\"yöntem\", \"süre\"], [[\"piksel döngüsü (NumPy)\", fmt_time(t_ref)], [\"vektörel\", fmt_time(t_vec)]])\n",
 "x1": "import numpy as np\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    cases = [(\"gaussian\", dict(mean=5.0, sigma=12.0)), (\"rayleigh\", dict(a=10.0, b=400.0)), (\"erlang\", dict(a=0.2, b=4)),\n             (\"exponential\", dict(a=0.1)), (\"uniform\", dict(a=-20.0, b=30.0))]\n    for kind, p in cases:\n        def t(kind=kind, p=p):\n            z = need(ns, \"make_noise\")(kind, (200000,), np.random.default_rng(1), **p)\n            ensure(z.shape == (200000,), \"şekil\")\n            est = need(ns, \"estimate\")(kind, z)\n            for k, v in p.items():\n                ensure(abs(est[k] - v) < 0.03 * abs(v) + 0.3, f\"{k}: {est[k]:.3f}, gerçek {v}\")\n            return \", \".join(f\"{k} = {est[k]:.3f}\" for k in p)\n        R.test(f\"{kind}: üret ve kestir\", t)\n\n    def spt():\n        img = data.camera()\n        out = need(ns, \"add_salt_pepper\")(img, 0.05, 0.08, np.random.default_rng(2))\n        ensure(abs((out == 255).mean() - (img == 255).mean() - 0.05) < 0.005 and abs((out == 0).mean() - (img == 0).mean() - 0.08) < 0.006, \"oranlar\")\n        ensure(img.dtype == out.dtype and not np.shares_memory(img, out), \"kopya, uint8\")\n    R.test(\"add_salt_pepper\", spt)\n",
 "x2": "import numpy as np\nfrom scipy import ndimage\nfrom skimage import data\nfrom labcheck import close, no_loops, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    img = data.camera()[200:260, 200:260]\n    f = img.astype(np.float64) + 1\n    R.test(\"geometric_mean\", lambda: close(need(ns, \"geometric_mean\")(img, 3), ndimage.generic_filter(f, lambda w: np.exp(np.log(w).mean()), 3, mode=\"nearest\") - 1, atol=1e-8))\n    R.test(\"harmonic_mean\", lambda: close(need(ns, \"harmonic_mean\")(img, 3), ndimage.generic_filter(f, lambda w: 9 / (1 / w).sum(), 3, mode=\"nearest\") - 1, atol=1e-8))\n    R.test(\"contraharmonic Q = 1.5\", lambda: close(need(ns, \"contraharmonic\")(img, 3, 1.5), ndimage.generic_filter(f, lambda w: (w ** 2.5).sum() / (w ** 1.5).sum(), 3, mode=\"nearest\") - 1, atol=1e-6))\n\n    def sign():\n        clean = data.camera()\n        rng = np.random.default_rng(0)\n        pepper = clean.copy(); pepper[rng.random(clean.shape) < 0.1] = 0\n        salt = clean.copy(); salt[rng.random(clean.shape) < 0.1] = 255\n        e = lambda a: np.abs(a - clean).mean()\n        rows = [[\"biber\", f\"{e(ns['contraharmonic'](pepper, 3, 1.5)):.1f}\", f\"{e(ns['contraharmonic'](pepper, 3, -1.5)):.1f}\"],\n                [\"tuz\", f\"{e(ns['contraharmonic'](salt, 3, 1.5)):.1f}\", f\"{e(ns['contraharmonic'](salt, 3, -1.5)):.1f}\"]]\n        ensure(e(ns[\"contraharmonic\"](pepper, 3, 1.5)) < e(ns[\"contraharmonic\"](pepper, 3, -1.5)), \"biberde Q > 0 daha iyi\")\n        ensure(e(ns[\"contraharmonic\"](salt, 3, -1.5)) < e(ns[\"contraharmonic\"](salt, 3, 1.5)), \"tuzda Q < 0 daha iyi\")\n        R.table(\"Ortalama mutlak hata\", [\"gürültü\", \"Q = +1.5\", \"Q = −1.5\"], rows)\n    R.test(\"Q'nun işareti: biber için +, tuz için −\", sign)\n",
 "x3": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom skimage.metrics import structural_similarity\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    clean = data.camera()\n\n    def table():\n        rows = []\n        for p in (0.05, 0.2, 0.4):\n            rng = np.random.default_rng(int(p * 100))\n            noisy = clean.copy(); u = rng.random(clean.shape); noisy[u < p / 2] = 0; noisy[(u >= p / 2) & (u < p)] = 255\n            t = need(ns, \"denoise_table\")(clean, noisy)\n            ensure(set(t) == {\"box5\", \"gauss7\", \"median5\", \"cwmedian5\"}, \"dört anahtar\")\n            close(t[\"median5\"], cv2.PSNR(clean, cv2.medianBlur(noisy, 5)), atol=1e-6, what=\"median5\")\n            rows.append([f\"%{int(p * 100)}\"] + [f\"{t[k]:.2f}\" for k in (\"box5\", \"gauss7\", \"median5\", \"cwmedian5\")])\n        R.table(\"Tuz-biber, PSNR (dB)\", [\"gürültü\", \"kutu 5\", \"Gauss 7\", \"median 5\", \"CW median 5\"], rows,\n                note=\"Düşük yoğunlukta merkez ağırlıklı median ayrıntıyı daha iyi korur; yüksek yoğunlukta merkezdeki gürültüyü de koruduğu için geride kalır.\")\n    R.test(\"filtre karşılaştırma tablosu\", table)\n\n    def fool():\n        out = need(ns, \"fool_psnr\")(clean)\n        p = cv2.PSNR(clean, out)\n        s = structural_similarity(clean, out, data_range=255)\n        ensure(out.shape == clean.shape and out.dtype == np.uint8, \"aynı boyut, uint8\")\n        ensure(p < 25 and s > 0.9, f\"PSNR {p:.2f} dB, SSIM {s:.3f}: PSNR < 25 ve SSIM > 0.9 bekleniyordu\")\n        return f\"PSNR {p:.2f} dB, SSIM {s:.3f}: görsel olarak aynı, PSNR'a göre 'kötü'\"\n    R.test(\"PSNR'ı kandır (SSIM yüksek kalsın)\", fool)\n"
}

In [ ]:
import importlib, labcheck, checks_m9_5_restoration as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Bozulma modeli:** $g(x,y)=h*f(x,y)+\eta(x,y)$. Bu modülde yalnız gürültü ($h$ özdeşlik): $g=f+\eta$.

**Gürültü yoğunlukları** ($z$ gri seviye):

| model | yoğunluk | nerede |
|---|---|---|
| Gauss | $\frac{1}{\sqrt{2\pi}\sigma}e^{-(z-\bar z)^2/2\sigma^2}$ | elektronik devre, sensör gürültüsü |
| Rayleigh | $\frac2b(z-a)e^{-(z-a)^2/b}$, $z\ge a$ | menzil görüntüleme |
| Erlang (Gamma) | $\frac{a^bz^{b-1}}{(b-1)!}e^{-az}$ | lazer görüntüleme |
| üstel | $ae^{-az}$ | lazer görüntüleme |
| düzgün | $\frac{1}{b-a}$, $a\le z\le b$ | simülasyon |
| tuz-biber | $P_s$ olasılıkla 255, $P_p$ olasılıkla 0 | hatalı pikseller, iletim |

Gürültü parametreleri görüntünün **düz bir bölgesinden** ortalama ve varyansla kestirilir.

**Ortalama filtreler** ($S_{xy}$ pencere, $mn$ piksel): aritmetik $\frac1{mn}\sum g$, geometrik $(\prod g)^{1/mn}$, harmonik $mn/\sum\frac1g$, kontraharmonik $\sum g^{Q+1}/\sum g^Q$ ($Q>0$ biberi, $Q<0$ tuzu temizler).

**median filtre:** penceredeki değerlerin medyanı. **Merkez ağırlıklı median:** merkez piksel $w$ kez sayılır; ayrıntıyı daha iyi korur.

**Uyarlamalı ortalama** (yerel gürültü azaltma): $\hat f=g-\frac{\sigma_\eta^2}{\sigma_S^2}(g-m_S)$; $\sigma_\eta^2>\sigma_S^2$ ise oran 1'e kırpılır (çıkış yerel ortalama).

**Uyarlamalı median:**

- *Seviye A:* $z_{min}<z_{med}<z_{max}$ ise B'ye git; değilse pencereyi büyüt, $S_{max}$'a ulaşıldıysa $z_{med}$ ver.
- *Seviye B:* $z_{min}<z_{xy}<z_{max}$ ise $z_{xy}$ (piksel gürültü değil), değilse $z_{med}$.

**PSNR:** $10\log_{10}\frac{R^2}{\text{MSE}}$, 8 bit için $R=255$.

## 1 · Elle

Tek bir 255'lik tuz pikseli aritmetik ortalamayı 12'den 38'e çekiyor; median'ı hiç etkilemiyor.

Uyarlamalı median'da merkez piksel 255 = $z_{max}$, yani gürültü sayılıp yerine median konur.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**3 × 3 pencere $\{10, 12, 11, 255, 13, 0, 12, 11, 14\}$ (merkez 13)**

- `mean`: Aritmetik ortalama (tamsayıya yuvarla)
- `med`: Median
- `cwm`: Merkez 3 kez sayılınca median (11 eleman)

**Uyarlamalı median: $z_{min}=0$, $z_{max}=255$, $z_{med}=12$, $z_{xy}=255$**

- `lvlA`: Seviye A geçilir mi? ($z_{min}<z_{med}<z_{max}$; evet 1, hayır 0)
- `amout`: Çıkış (Seviye B)

**Uyarlamalı ortalama $\hat f=g-\frac{\sigma_\eta^2}{\sigma_S^2}(g-m_S)$, $[0,1]$ ölçeğinde**

- `am1`: $g=0.6$, $m_S=0.5$, $\sigma_\eta^2=0.01$, $\sigma_S^2=0.04$
- `am2`: Aynı, ama $\sigma_S^2=0.005<\sigma_\eta^2$ (oran 1'e kırpılır)

**PSNR ve diğer**

- `psnr`: 8 bit, MSE = 100: PSNR (dB)
- `geo`: Geometrik ortalama $\{1, 4, 16\}$

In [ ]:
cevaplar = dict(
    mean=None,
    med=None,
    cwm=None,
    lvlA=None,
    amout=None,
    am1=None,
    am2=None,
    psnr=None,
    geo=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"mean":[38,0],"med":[12,0],"cwm":[12,0],"lvlA":[1,0],"amout":[12,0],"am1":[0.575,0],"am2":[0.5,0],"psnr":[28.1308,0.001],"geo":[4,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python; `uint8` değerli liste listesi; kenarlar **kopyalama** (`BORDER_REPLICATE`, `cv2.medianBlur`'un kuralı).

- `median_filter(img, k)` — `k × k` (tek).
- `cw_median(img, k, w)` — merkez `w` kez sayılır (pencereye `w-1` kopya ekle).
- `adaptive_median(img, smax=7)` — pencere 3'ten başlar, 2'şer büyür.

Kontrol `cv2.medianBlur` ile birebir ve uyarlamalı median'ı başvuru uygulamasıyla karşılaştırır.

In [ ]:
%%writefile stage_b.py
def median_filter(img, k):
    ...


def cw_median(img, k, w):
    ...


def adaptive_median(img, smax=7):
    ...


img = [[10, 12, 11], [255, 13, 0], [12, 11, 14]]
print(median_filter(img, 3)[1][1], cw_median(img, 3, 3)[1][1])

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, **döngü yok**. Kayan pencere: `np.lib.stride_tricks.sliding_window_view(np.pad(img, r, mode="edge"), (k, k))` → `(H, W, k, k)`.

- `median_np(img, k)` — `cv2.medianBlur` ile birebir.
- `adaptive_mean(img_u8, noise_var, k=5)` — görüntüyü $[0,1]$'e normalize et (`float32`), yerel ortalama/varyans (kopyalama kenarı), oran $\min(\sigma_\eta^2/\sigma_S^2, 1)$, sonucu `uint8`'e geri çevir (`np.rint`, kırp).
- `psnr(a, b)` — `cv2.PSNR` ile aynı. OpenCV sıfıra bölmeyi önlemek için $20\log_{10}\frac{255}{\sqrt{\text{MSE}}+\varepsilon}$ hesaplar ($\varepsilon$ = `np.finfo(np.float64).eps`); özdeş görüntülerde sonuç ≈ 361.2 dB.

In [ ]:
%%writefile stage_c.py
import numpy as np
import cv2


def median_np(img, k):
    ...


def adaptive_mean(img_u8, noise_var, k=5):
    ...


def psnr(a, b):
    ...


from skimage import data
img = data.camera()
print(psnr(img, median_np(img, 5)))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Küçük bir artık (residual) gürültü giderici:

- `Denoiser(c=32, depth=4)` — `Conv3×3 → ReLU → (Conv3×3 → ReLU) × (depth-2) → Conv3×3`, hepsi `padding=1`; ağ **gürültüyü** tahmin eder, çıkış `x - net(x)`.
- `train_denoiser(img, sigma, steps=500, batch=16, patch=48, seed=0)` — `img` [0,1] float tensör; her adımda rastgele yamalar, $\sigma$ Gauss gürültüsü ekle, MSE (Adam, lr = 1e-3). Modeli döndür.

Kontrol, `camera` ile eğitip görülmemiş `coins` görüntüsünde $\sigma=25/255$ için PSNR'ı Gauss bulanıklaştırma ile karşılaştırır.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn.functional as F
from torch import nn


class Denoiser(nn.Module):
    def __init__(self, c=32, depth=4):
        super().__init__()
        ...

    def forward(self, x):
        ...


def train_denoiser(img, sigma, steps=500, batch=16, patch=48, seed=0):
    ...


from skimage import data
img = torch.tensor(data.camera(), dtype=torch.float32) / 255
m = train_denoiser(img, 25 / 255, steps=20)
print(m(img[None, None, :64, :64]).shape)

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`adaptive_median_vec(img, smax=7)` — `img: (H, W)` uint8 tensör. Döngüsüz fikir: her pencere boyutu ($3,5,7$) için $z_{min}, z_{med}, z_{max}$'ı `unfold` ile **hepsini birden** hesapla; sonra küçükten büyüğe "ilk Seviye A'yı geçen boyut"u maskelerle seç. Pencere boyutları üzerinde sabit uzunluklu bir `for` (3 tekrar) serbest; piksel döngüsü yok. Kenarlar kopyalama.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F


def adaptive_median_vec(img, smax=7):
    """img (H, W) uint8 tensör. Piksel döngüsü yok."""
    ...


img = torch.randint(0, 256, (16, 16), dtype=torch.uint8)
print(adaptive_median_vec(img).shape)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Gürültü modelleri: üretme ve düz bir bölgeden parametre kestirimi

- `make_noise(kind, shape, rng, **p)` — `kind`: `"gaussian"` (`mean`, `sigma`), `"rayleigh"` (`a`, `b`), `"erlang"` (`a`, `b`: tamsayı), `"exponential"` (`a`), `"uniform"` (`a`, `b`). NumPy `rng`'nin dağılımlarıyla; Rayleigh için $z=a+\sqrt{-b\ln(1-U)}$.
- `estimate(kind, samples)` — moment yöntemiyle parametreleri döndür (sözlük). Rayleigh: $\bar z=a+\sqrt{\pi b/4}$, $\sigma^2=b(4-\pi)/4$; Erlang: $\bar z=b/a$, $\sigma^2=b/a^2$; düzgün: $\bar z=(a+b)/2$, $\sigma^2=(b-a)^2/12$.
- `add_salt_pepper(img, ps, pp, rng)` — `uint8`.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def make_noise(kind, shape, rng, **p):
    ...


def estimate(kind, samples):
    ...


def add_salt_pepper(img, ps, pp, rng):
    ...


rng = np.random.default_rng(0)
z = make_noise("rayleigh", (100000,), rng, a=10, b=400)
print(estimate("rayleigh", z))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Ortalama filtre ailesi: geometrik, harmonik, kontraharmonik

Kayan pencereyle (NumPy, döngüsüz, kenar kopyalama), `float64` hesap:

- `geometric_mean(img, k)` — log'ların ortalamasının üssü; sıfırları önlemek için $g+1$ üzerinde hesapla, sonra 1 çıkar.
- `harmonic_mean(img, k)` — $mn/\sum 1/(g+1)$, sonra 1 çıkar.
- `contraharmonic(img, k, Q)` — $\sum g^{Q+1}/\sum g^Q$ ($g+1$ üzerinde, sonra 1 çıkar).

Deney: biberli görüntüde $Q=1.5$ temizler, $Q=-1.5$ kötüleştirir; tuzluda tersi.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def geometric_mean(img, k):
    ...


def harmonic_mean(img, k):
    ...


def contraharmonic(img, k, Q):
    ...


from skimage import data
print(geometric_mean(data.camera(), 3).mean())

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · PSNR deneyi: filtreler, gürültü seviyeleri ve PSNR'ı 'kandırmak'

Alıştırmanın PSNR deneyleri (OpenCV filtreleri serbest):

- `denoise_table(clean, noisy)` → `{"box5": psnr, "gauss7": psnr, "median5": psnr, "cwmedian5": psnr}`; kutu 5 × 5, Gauss 7 × 7 (σ otomatik), `cv2.medianBlur` 5, kendi merkez ağırlıklı median'ın ($w=3$, c'deki pencerelerle).
- `fool_psnr(img)` — görsel olarak aynı görünen ama `cv2.PSNR(img, out) < 25` olan bir görüntü üret (ipucu: çok basit bir dönüşüm). Kontrol SSIM'in 0.9'un üstünde kaldığını da sınar.

In [ ]:
%%writefile extra_x3.py
import numpy as np
import cv2


def denoise_table(clean, noisy):
    ...


def fool_psnr(img):
    ...


from skimage import data
print(fool_psnr(data.camera()).shape)

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: DnCNN: residual öğrenmeyle derin gürültü giderici

[Zhang, Zuo, Chen, Meng, Zhang — Beyond a Gaussian Denoiser: Residual Learning of Deep CNN for Image Denoising (IEEE TIP 2017)](https://arxiv.org/abs/1608.03981)

17 katmanlı DnCNN'i (conv + BN + ReLU, gürültüyü tahmin eden residual çıkış) sıfırdan uygula; BSD400 benzeri bir eğitim kümesinde (yoksa scikit-image ve kendi çektiğin fotoğraflardan 40×40 yamalar) σ = 25 için eğit ve BSD68/Set12 tarzı bir test setinde klasik filtrelerle (Gauss, median, bilateral, NLM — M9.12) karşılaştır.


**Adımlar**

1. Yama çıkarımı ve veri artırma (8 yönlü dönüş/yansıma).
2. DnCNN-S mimarisi: ilk katman conv+ReLU, orta 15 katman conv+BN+ReLU, son katman conv; kayıp $\frac12\lVert R(y)-(y-x)\rVert^2$.
3. Kör (σ ∈ [0, 55]) sürümü de eğit (DnCNN-B) ve sabit σ ile karşılaştır.
4. Her yöntem için PSNR ve SSIM tablosu ve görsel karşılaştırma.

**Kabul ölçütleri**

- σ = 25'te test PSNR'ı Gauss bulanıklaştırmadan en az 3 dB, NLM'den en az 0.5 dB yüksek.
- Residual (gürültü tahmini) ve doğrudan (temiz görüntü tahmini) çıkış karşılaştırması raporlanmış.
- BN'li ve BN'siz eğitim eğrileri karşılaştırılmış.

## Kaynaklar

- [Gonzalez & Woods — Digital Image Processing, Bölüm 5.1–5.3](https://www.imageprocessingplace.com/) — ©
- [OpenCV — medianBlur, PSNR](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html) — Apache-2.0
- [Hwang & Haddad — Adaptive median filters: new algorithms and results (IEEE TIP 1995)](https://doi.org/10.1109/83.370679) — ©
- [Zhang ve ark. — Beyond a Gaussian Denoiser: Residual Learning of Deep CNN for Image Denoising (DnCNN)](https://arxiv.org/abs/1608.03981) — arXiv